# 02 — Cleaning and Joining

This notebook joins the four processed datasets into a single analytical table. The main challenges:

1. **YJB name matching** — the YJB dataset has no LA codes, only names. 22 of 154 names don't match DfE names directly due to punctuation differences, boundary changes, and footnote annotations.
2. **Temporal alignment** — DfE data runs to academic year-end (August); YJB to calendar year-end (December). We align them with a consistent offset: YJB year *n* maps to DfE academic year *n-1/n*.
3. **Suppressed values** — small LAs have some exclusion rates suppressed. We document the scale and decide how to handle them.
4. **Boundary changes** — several LAs changed boundaries between 2019 (IMD) and 2023/24 (exclusions). We handle the main cases explicitly.

Output: `data/processed/analytical_table.csv` — one row per LA per year, ready for notebook 03.

---

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path('../data/processed')

exc = pd.read_csv(PROCESSED / 'exclusions_sen_la.csv')
yjb = pd.read_csv(PROCESSED / 'yjb_fte_la.csv')
imd = pd.read_csv(PROCESSED / 'imd_la.csv')
sen = pd.read_csv(PROCESSED / 'sen_needs_la.csv')

print(f'exclusions_sen_la:  {exc.shape}')
print(f'yjb_fte_la:         {yjb.shape}')
print(f'imd_la:             {imd.shape}')
print(f'sen_needs_la:       {sen.shape}')

exclusions_sen_la:  (7748, 8)
yjb_fte_la:         (1705, 4)
imd_la:             (317, 3)
sen_needs_la:       (153, 24)


**Boundary change note:** Three LA names appear twice in the DfE data with different ONS codes — Buckinghamshire, North Yorkshire, and Somerset each have both a legacy county council code (E10) and a newer unitary authority code (E06) covering the same geography. We drop the legacy codes, keeping only the current unitary codes.

In [11]:
# Drop legacy county council codes superseded by unitary authority codes
# Buckinghamshire E10000002 -> E06000060
# North Yorkshire  E10000023 -> E06000065
# Somerset         E10000027 -> E06000066
exc = exc[~exc['new_la_code'].isin(['E10000002', 'E10000023', 'E10000027'])].copy()
print(f'LAs after dropping legacy codes: {exc["new_la_code"].nunique()}')

LAs after dropping legacy codes: 155


## Step 1 — Resolve YJB name → ONS code

132 of 154 YJB LA names match DfE names directly. The remaining 22 fall into four categories:

- **Regional totals** (8 rows) — drop, we only want individual LAs
- **Punctuation / minor spelling differences** (4) — simple string fixes
- **Pre-merger LAs** (3) — Bournemouth and Poole merged to BCP; old Northamptonshire split to North/West Northants. YJB carries both old and new rows during transition years.
- **New unitary authorities** (4) — Cumberland, Westmorland & Furness, North Yorkshire, East Riding name variants
- **Footnote annotations** (several) — `[note 12]` suffixes added to boundary-changed LAs

In [12]:
# Build name mapping: YJB name -> DfE/ONS name
# Regional totals are dropped (mapped to None)

YJB_NAME_MAP = {
    # Regional totals — drop
    'Total East Midlands':             None,
    'Total London':                    None,
    'Total North East':                None,
    'Total North West':                None,
    'Total South East':                None,
    'Total South West':                None,
    'Total West Midlands':             None,
    'Total Yorkshire and the Humber':  None,

    # Punctuation / spelling
    'Durham':                          'County Durham',
    'Herefordshire':                   'Herefordshire, County of',
    'Newcastle-upon Tyne':             'Newcastle upon Tyne',
    'St Helens':                       'St. Helens',

    # Geography name variants
    'East Riding of Yorkshire and the Humber': 'East Riding of Yorkshire',
    'North Yorkshire and the Humber':          'North Yorkshire',
    'West Morland and Furness [note 12]':      'Westmorland and Furness',
    'Cumberland [note 12]':                    'Cumberland',

    # Boundary changes — pre-merger LAs: keep for years they existed,
    # map to successor for post-merger years
    'Bournemouth [note 12]':                       'Bournemouth, Christchurch and Poole',
    'Poole [note 12]':                             'Bournemouth, Christchurch and Poole',
    'Bournemouth, Christchurch and Poole [note 12]': 'Bournemouth, Christchurch and Poole',
    'North Northamptonshire [note 12]':            'North Northamptonshire',
    'West Northamptonshire [note 12]':             'West Northamptonshire',
    'Dorset [note 12]':                            'Dorset',
}

# Apply mapping — drop rows mapped to None
yjb['la_name_dfe'] = yjb['la_name'].map(lambda x: YJB_NAME_MAP.get(x, x))
yjb = yjb[yjb['la_name_dfe'].notna()].copy()

print(f'YJB rows after dropping regional totals: {len(yjb)}')
print(f'Unique LA names: {yjb["la_name_dfe"].nunique()}')

YJB rows after dropping regional totals: 1617
Unique LA names: 144


In [13]:
# Build LA name -> code lookup from exclusions data
la_lookup = (
    exc[['new_la_code', 'la_name']]
    .drop_duplicates()
    .set_index('la_name')['new_la_code']
)

# Join code onto YJB
yjb['la_code'] = yjb['la_name_dfe'].map(la_lookup)

# Check what's still unmatched
unmatched = yjb[yjb['la_code'].isna()]['la_name_dfe'].unique()
print(f'Still unmatched after mapping: {len(unmatched)}')
if len(unmatched):
    for n in sorted(unmatched):
        print(f'  {n}')

Still unmatched after mapping: 0


In [14]:
# Drop any remaining unmatched rows and keep only what we need
yjb_clean = yjb[yjb['la_code'].notna()][['la_code', 'la_name_dfe', 'yjb_year', 'fte_rate_per_100k']].copy()
yjb_clean = yjb_clean.rename(columns={'la_name_dfe': 'la_name'})

print(f'YJB clean shape: {yjb_clean.shape}')
yjb_clean.head()

YJB clean shape: (1617, 4)


,la_code,la_name,yjb_year,fte_rate_per_100k
0,E06000015,Derby,2014,543.958
1,E10000007,Derbyshire,2014,275.226
2,E06000016,Leicester,2014,578.026
3,E10000018,Leicestershire,2014,314.159
4,E10000019,Lincolnshire,2014,406.993


## Step 2 — Temporal Alignment

DfE academic year `202324` = September 2023 to August 2024.  
YJB calendar year `2023` = January to December 2023.

The closest match: YJB year *n* aligns to DfE year starting in *n* (e.g. YJB 2023 → DfE 202324). This is an approximation — there's a ~4 month overlap issue either way — but it's the standard approach in the literature and we document it explicitly.

DfE `time_period` format is `YYYYNN` (e.g. `202324`). We extract the start year to match against YJB year.

In [15]:
# Extract start year from DfE time_period (202324 -> 2023)
exc['dfe_start_year'] = (exc['time_period'] // 100).astype(int)

print('DfE years available:', sorted(exc['dfe_start_year'].unique()))
print('YJB years available:', sorted(yjb_clean['yjb_year'].dropna().unique()))
print()
print('Overlapping years (DfE start year = YJB year):')
dfe_years = set(exc['dfe_start_year'].unique())
yjb_years = set(yjb_clean['yjb_year'].dropna().astype(int).unique())
overlap = sorted(dfe_years & yjb_years)
print(overlap)

DfE years available: [2019, 2020, 2021, 2022, 2023, 2024]
YJB years available: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Overlapping years (DfE start year = YJB year):
[2019, 2020, 2021, 2022, 2023, 2024]


## Step 3 — Pivot Exclusions to Wide Format

Currently the exclusions data has one row per LA per year per SEN category. We pivot to one row per LA per year with separate columns for each SEN category's exclusion rate — this makes the join and subsequent analysis cleaner.

In [16]:
# Pivot SEN characteristic into columns
exc_wide = exc.pivot_table(
    index=['new_la_code', 'la_name', 'dfe_start_year'],
    columns='characteristic',
    values=['perm_excl_rate', 'susp_rate']
).reset_index()

# Flatten MultiIndex columns
exc_wide.columns = [
    f"{metric}_{char.lower().replace(' ', '_')}"
    if char else metric
    for metric, char in exc_wide.columns
]

print(f'Wide shape: {exc_wide.shape}')
print('Columns:', list(exc_wide.columns))

Wide shape: (903, 11)
Columns: ['new_la_code', 'la_name', 'dfe_start_year', 'perm_excl_rate_ehc_plan', 'perm_excl_rate_no_identified_sen', 'perm_excl_rate_sen_support', 'perm_excl_rate_unclassified', 'susp_rate_ehc_plan', 'susp_rate_no_identified_sen', 'susp_rate_sen_support', 'susp_rate_unclassified']


## Step 4 — Join All Datasets

Join order:
1. Start with exclusions (widest time coverage, LA codes as keys)
2. Join YJB on `la_code` + aligned year
3. Join IMD on `la_code` (static — same score for all years)
4. Join SEN needs on `la_code` (single year snapshot — same values for all years)

In [19]:
# Prepare YJB for join
yjb_join = yjb_clean.rename(columns={
    'la_code': 'new_la_code',
    'yjb_year': 'dfe_start_year'
})
yjb_join['dfe_start_year'] = yjb_join['dfe_start_year'].astype(int)

# Prepare IMD for join
imd_join = imd[['la_code', 'imd_avg_score']].rename(columns={'la_code': 'new_la_code'})

# Prepare SEN needs for join
sen_join = sen[['new_la_code', 'total_sen', 'pct_semh', 'pct_spld', 'pct_asd']]

# Join
df = exc_wide.copy()
df = df.merge(yjb_join[['new_la_code', 'dfe_start_year', 'fte_rate_per_100k']],
              on=['new_la_code', 'dfe_start_year'], how='left')
df = df.merge(imd_join, on='new_la_code', how='left')
df = df.merge(sen_join, on='new_la_code', how='left')

print(f'Analytical table shape: {df.shape}')
print(f'Unique LAs: {df["new_la_code"].nunique()}')
print(f'Years: {sorted(df["dfe_start_year"].unique())}')
print(f'Rows with YJB data: {df["fte_rate_per_100k"].notna().sum()}')
print(f'Rows with IMD data: {df["imd_avg_score"].notna().sum()}')


Analytical table shape: (921, 17)
Unique LAs: 155
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Rows with YJB data: 810
Rows with IMD data: 768


## Step 5 — Assess Missingness

Before saving, understand what's missing and why.

In [20]:
# Overall missingness
print('Missing value summary:')
missing = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(1)
summary = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print(summary[summary['missing'] > 0].to_string())

Missing value summary:
                             missing   pct
perm_excl_rate_unclassified      919  99.8
susp_rate_unclassified           919  99.8
fte_rate_per_100k                111  12.1
imd_avg_score                    153  16.6
total_sen                          6   0.7
pct_semh                           6   0.7
pct_spld                           6   0.7
pct_asd                            6   0.7


In [22]:
# Which LAs are missing YJB data entirely?
la_yjb_coverage = df.groupby('new_la_code')['fte_rate_per_100k'].apply(lambda x: x.notna().mean())
no_yjb = la_yjb_coverage[la_yjb_coverage == 0].index
print(f'LAs with no YJB data at all: {len(no_yjb)}')
if len(no_yjb):
    print(df[df['new_la_code'].isin(no_yjb)]['la_name'].unique())

LAs with no YJB data at all: 14
['Rutland' 'Peterborough' 'Luton' 'Southend-on-Sea' 'Thurrock'
 'Isles of Scilly' 'Bedford' 'Central Bedfordshire' 'City of London'
 'Cambridgeshire' 'Essex' 'Hertfordshire' 'Norfolk' 'Suffolk']


In [23]:
# Which LAs are missing IMD data?
# IMD 2019 uses 2019 boundaries — some newer unitary authorities won't match
no_imd = df[df['imd_avg_score'].isna()]['la_name'].unique()
print(f'LAs missing IMD score: {len(no_imd)}')
for n in sorted(no_imd):
    print(f'  {n}')

LAs missing IMD score: 30
  Buckinghamshire
  Cambridgeshire
  Cumberland
  Cumbria
  Derbyshire
  Devon
  East Sussex
  Essex
  Gloucestershire
  Hampshire
  Hertfordshire
  Kent
  Lancashire
  Leicestershire
  Lincolnshire
  Norfolk
  North Northamptonshire
  North Yorkshire
  Northamptonshire
  Nottinghamshire
  Oxfordshire
  Somerset
  Staffordshire
  Suffolk
  Surrey
  Warwickshire
  West Northamptonshire
  West Sussex
  Westmorland and Furness
  Worcestershire


In [24]:
# Suppressed exclusion rates — how much data is missing per SEN category?
excl_cols = [c for c in df.columns if c.startswith('perm_excl_rate_')]
print('Suppressed exclusion rates per SEN category:')
for col in excl_cols:
    n_missing = df[col].isna().sum()
    print(f'  {col}: {n_missing} ({n_missing/len(df)*100:.1f}%)')

Suppressed exclusion rates per SEN category:
  perm_excl_rate_ehc_plan: 0 (0.0%)
  perm_excl_rate_no_identified_sen: 0 (0.0%)
  perm_excl_rate_sen_support: 0 (0.0%)
  perm_excl_rate_unclassified: 919 (99.8%)


## Step 6 — Core Analytical Subset

For the main correlation analysis we need rows that have:
- An exclusion rate for SEN support (our primary exposure variable)
- A YJB first-time entrant rate (our outcome variable)
- An IMD score (our control variable)

We retain the full table for reference but flag which rows are usable for the core analysis.

In [25]:
# Identify the SEN support exclusion rate column name
sen_support_col = [c for c in df.columns if 'sen_support' in c and 'perm_excl' in c]
print('SEN support exclusion rate column(s):', sen_support_col)

if sen_support_col:
    core_col = sen_support_col[0]
    df['usable_for_core_analysis'] = (
        df[core_col].notna() &
        df['fte_rate_per_100k'].notna() &
        df['imd_avg_score'].notna()
    )
    print(f'\nRows usable for core analysis: {df["usable_for_core_analysis"].sum()} of {len(df)}')
    print(f'Unique LAs with at least 1 usable row: {df[df["usable_for_core_analysis"]]["new_la_code"].nunique()}')

SEN support exclusion rate column(s): ['perm_excl_rate_sen_support']

Rows usable for core analysis: 689 of 921
Unique LAs with at least 1 usable row: 116


In [26]:
df.to_csv(PROCESSED / 'analytical_table.csv', index=False)
print(f'Saved: analytical_table.csv  {df.shape}')
print('\nColumn list:')
for c in df.columns:
    print(f'  {c}')

Saved: analytical_table.csv  (921, 18)

Column list:
  new_la_code
  la_name
  dfe_start_year
  perm_excl_rate_ehc_plan
  perm_excl_rate_no_identified_sen
  perm_excl_rate_sen_support
  perm_excl_rate_unclassified
  susp_rate_ehc_plan
  susp_rate_no_identified_sen
  susp_rate_sen_support
  susp_rate_unclassified
  fte_rate_per_100k
  imd_avg_score
  total_sen
  pct_semh
  pct_spld
  pct_asd
  usable_for_core_analysis


## Summary

The analytical table joins all four datasets on ONS LA code. Key findings from this notebook:

- YJB name resolution required 14 explicit mappings plus dropping 8 regional subtotal rows
- Temporal alignment: YJB calendar year *n* maps to DfE academic year *n/n+1*
- IMD 2019 won't cover the newest unitary authorities (post-2019 boundary changes) — small number of LAs affected, noted but not imputed
- Suppressed exclusion rates affect small LAs — documented above

**Next:** Notebook 03 — exploratory analysis. Distributions, LA-level maps, and the first look at whether the expected pattern is actually in the data.